In [6]:
import geopandas as gpd

PARKS_FILE = "parks.geojson"
HOODS_FILE = "neighborhoods.geojson"

parks = gpd.read_file(PARKS_FILE)
hoods = gpd.read_file(HOODS_FILE)
print("Neighborhood file columns:", list(hoods.columns), "\n")

hood_col = None
for candidate in ["hood", "neighborhood", "neighbor", "name"]:
    for col in hoods.columns:
        if candidate in col.lower() and col.lower() != "geometry":
            hood_col = col
            break
    if hood_col:
        break
if hood_col is None:
    raise SystemExit("Couldn't find the neighborhood name column. "
                     "Set hood_col manually using the list above.")
print(f"Using neighborhood column: {hood_col!r}\n")

CRS = "EPSG:2272"
parks = parks.to_crs(CRS)
hoods = hoods.to_crs(CRS)

parks["geometry"] = parks.geometry.buffer(0)
hoods["geometry"] = hoods.geometry.buffer(0)
parks_merged = parks[parks.geometry.notna()][["geometry"]].dissolve()

pieces = gpd.overlay(parks_merged, hoods[[hood_col, "geometry"]],
                     how="intersection")
pieces["acres"] = pieces.geometry.area / 43560  # sq ft -> acres
totals = pieces.groupby(hood_col)["acres"].sum()

totals = totals.reindex(hoods[hood_col].unique(), fill_value=0)
totals = totals.sort_values(ascending=False)

print(f"{'Rank':<5}{'Neighborhood':<35}{'Park acres':>12}")
print("-" * 52)
for rank, (hood, acres) in enumerate(totals.items(), start=1):
    print(f"{rank:<5}{str(hood):<35}{acres:>12,.2f}")


    #Printing the conclusion

paragraph = (
            "Conclusion:" 
            "Basing this purely off the acres of land and not the subjective quality of the parks:"
            "The two neighborhoods with 500+ acres of land will get a 10/10. Highland Park and Hay will get a 9/10."
            "Any neighborhood remaining over 100 acres of land will get an 8/10." 
            "The rest of neighborhoods over 50 acres will get a 7/10." 
            "We are outside of the top of the top 15 now, so we won’t specify any more scores." 
            "If (and this likely) there is a specific neighborhood that excels in other categories"
            "but was included in the top neighborhood for parks, a score will be determined."
    
)

print()
print(paragraph)

Neighborhood file columns: ['OBJECTID', 'fid_blockg', 'statefp10', 'countyfp10', 'tractce10', 'blkgrpce10', 'geoid10', 'namelsad10', 'mtfcc10', 'funcstat10', 'aland10', 'awater10', 'intptlat10', 'intptlon10', 'shape_leng', 'fid_neighb', 'perimeter', 'neighbor_', 'neighbor_i', 'hood', 'hood_no', 'acres', 'sqmiles', 'dpwdiv', 'unique_id', 'sectors', 'shape_le_1', 'shape_ar_1', 'page_number', 'plannerassign', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date', 'temp', 'pghdb_sde_Neighborhood_2010_are', 'GlobalID', 'Shape__Area', 'Shape__Length', 'geometry'] 

Using neighborhood column: 'hood'

Rank Neighborhood                         Park acres
----------------------------------------------------
1    Squirrel Hill South                      748.73
2    Hays                                     627.53
3    Highland Park                            356.14
4    Perry North                              260.05
5    Duquesne Heights                         125.55
6    Swissh